In [1]:
# 04_embed_openai.ipynb (updated)
import json
import numpy as np
from pathlib import Path
from openai import OpenAI
import os

# Initialize client
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

CHUNKS_FILE = Path("data/chunks.jsonl")
EMBED_FILE = Path("embeddings/openai_embeddings.npy")
META_FILE = Path("embeddings/metadata_openai.json")

texts, metadata = [], []

with CHUNKS_FILE.open(encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        texts.append(rec["text"])
        metadata.append({
            "chunk_id": rec["doc_id"],
            "lang": rec["lang"],
            "disease_name_en": rec.get("disease_name_en"),
            "disease_name_si": rec.get("disease_name_si")
        })

embeddings = []

# Batch request
for i in range(0, len(texts), 50):
    batch = texts[i:i+50]
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend([e.embedding for e in resp.data])

embeddings = np.array(embeddings)
EMBED_FILE.parent.mkdir(exist_ok=True)
np.save(EMBED_FILE, embeddings)

with META_FILE.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved OpenAI embeddings: {EMBED_FILE}")
print(f"Saved metadata: {META_FILE}")


Saved OpenAI embeddings: embeddings\openai_embeddings.npy
Saved metadata: embeddings\metadata_openai.json
